# Chapter 1.1 — Library of Congress: Unidentified Photographs

**Dataset 3 of 3.** This notebook queries the Library of Congress JSON API for photographs whose subject is *unidentified*. These are images of human beings — soldiers, families, workers, strangers — whose names and contexts have been lost. The cataloguers admit defeat in their own metadata: *subject: unidentified*.

This is the visual stratum of the project's archive. Every image is a face the institutional record cannot put a name to.

**Target:** ≥ 250 records with a downloadable thumbnail and at least one line of text (title or description).

**Method:** the LoC search endpoint `https://www.loc.gov/photos/?fo=json` accepts a query string and returns paginated JSON. We restrict to images that have an `image_url` array (so we can actually download a thumbnail). For each record we save title, date, description, subjects, the LoC permalink, and the smallest available image (low-res *is* the point — these are visual remains, not high-resolution archives).

**Output:** `data/raw/loc/loc_records.csv` + downloaded thumbnails in `data/raw/loc/images/`.

**Fallback:** if the LoC API is unavailable or returns too few results, the last cell points at a Wikimedia Commons `Category:Unidentified_people` query as a backup.

In [1]:
import sys, os, time, json, re
from pathlib import Path

# Robust project-root finder: works whether JupyterLab launched the kernel
# from the project folder or from anywhere else on disk (e.g. Desktop).
def _find_project_root(marker="sa_utils.py"):
    p = Path.cwd().resolve()
    for c in [p] + list(p.parents):
        if (c / marker).exists(): return c
    cowork = Path.home() / "Library/Application Support/Claude/local-agent-mode-sessions"
    if cowork.exists():
        for hit in cowork.rglob(marker):
            return hit.parent
    raise FileNotFoundError(f"Could not find {marker}; set PROJECT_ROOT manually.")
PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from sa_utils import load_env, make_session, polite_sleep, DATA_RAW, safe_filename
import pandas as pd

load_env()
OUT_DIR = DATA_RAW / "loc"
IMG_DIR = OUT_DIR / "images"
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUT_DIR}")
print(f"Images dir:   {IMG_DIR}")

.env loaded. OPENAI_API_KEY present: True
Project root: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs
Output dir:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/loc
Images dir:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/loc/images


## 1. Paginated LoC search

The query is the literal word `unidentified`. We hit `loc.gov/photos/` (rather than the whole catalogue) to restrict the results to the Prints & Photographs Division. `c=50` per page; we paginate until we have enough.

The LoC JSON response contains rich metadata. We keep only the fields we need downstream.

In [2]:
session = make_session()

LOC_BASE = "https://www.loc.gov/photos/"

def search_loc(query: str = "unidentified", want: int = 300, per_page: int = 50, max_pages: int = 30):
    """Yield record dicts from paginated LoC search until `want` is reached or pages run out."""
    records = []
    for page in range(1, max_pages + 1):
        params = {"q": query, "fo": "json", "c": per_page, "sp": page}
        r = session.get(LOC_BASE, params=params, timeout=30)
        if r.status_code != 200:
            print(f"  page {page}: HTTP {r.status_code} — stopping")
            break
        try:
            data = r.json()
        except ValueError:
            print(f"  page {page}: non-JSON response — stopping")
            break
        results = data.get("results", [])
        if not results:
            print(f"  page {page}: empty — stopping")
            break
        records.extend(results)
        print(f"  page {page}: +{len(results)} (running total {len(records)})")
        if len(records) >= want:
            break
        polite_sleep(1.0)
    return records[:want]

raw = search_loc(query="unidentified", want=300)
print(f"\nRAW RECORDS COLLECTED: {len(raw)}")

  page 1: +50 (running total 50)
  page 2: +50 (running total 100)
  page 3: +50 (running total 150)
  page 4: +50 (running total 200)
  page 5: +50 (running total 250)
  page 6: +50 (running total 300)

RAW RECORDS COLLECTED: 300


## 2. Normalise records + download thumbnails

LoC `image_url` is an ordered list from smallest to largest. We pick the smallest available (low-resolution is the project's aesthetic; also reduces download time). If a record lacks `image_url`, we skip it — we need a visual to feed the image-vectorisation step.

In [3]:
def pick_smallest_image(item: dict) -> str | None:
    urls = item.get("image_url") or []
    if not urls:
        return None
    return urls[0]

def first_or_blank(x):
    if isinstance(x, list):
        return x[0] if x else ""
    return x if x is not None else ""

def join_list(x, sep=" | "):
    if isinstance(x, list):
        return sep.join(str(v) for v in x if v)
    return str(x or "")

rows = []
for i, item in enumerate(raw, 1):
    img_url = pick_smallest_image(item)
    if not img_url:
        continue
    rec_id = item.get("id", "").rstrip("/").split("/")[-1] or f"loc_{i:04d}"
    rec = {
        "loc_id":       rec_id,
        "title":        first_or_blank(item.get("title")),
        "date":         first_or_blank(item.get("date")),
        "description":  join_list(item.get("description")),
        "subject":      join_list(item.get("subject")),
        "location":     join_list(item.get("location")),
        "permalink":    item.get("id"),
        "image_url":    img_url,
    }
    rows.append(rec)

print(f"Records with downloadable image: {len(rows)}")

Records with downloadable image: 298


In [4]:
# ---- Download thumbnails ----
from PIL import Image
from io import BytesIO

kept = []
for i, rec in enumerate(rows, 1):
    fname = f"{rec['loc_id']}_{safe_filename(rec['title'], 40)}.jpg"
    fpath = IMG_DIR / fname
    if not fpath.exists():
        try:
            r = session.get(rec["image_url"], timeout=30)
        except Exception:
            continue
        if r.status_code != 200 or not r.content:
            continue
        try:
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img.save(fpath, format="JPEG", quality=85)
        except Exception:
            continue
        polite_sleep(0.4)
    rec["local_path"] = str(fpath.relative_to(PROJECT_ROOT))
    kept.append(rec)
    if i % 25 == 0:
        print(f"   {i:3d}/{len(rows)}  kept so far: {len(kept)}")

print(f"\nTOTAL IMAGES DOWNLOADED: {len(kept)}")

    25/298  kept so far: 25
    50/298  kept so far: 45
    75/298  kept so far: 70
   100/298  kept so far: 95
   150/298  kept so far: 143
   200/298  kept so far: 184
   225/298  kept so far: 204
   250/298  kept so far: 225
   275/298  kept so far: 246

TOTAL IMAGES DOWNLOADED: 265


In [5]:
df = pd.DataFrame(kept)
df["combined_text"] = (df["title"].fillna("") + ". " + df["description"].fillna("") + " " + df["subject"].fillna("")).str.strip()
df = df[df["combined_text"].str.len() >= 10].reset_index(drop=True)

csv_path = OUT_DIR / "loc_records.csv"
json_path = OUT_DIR / "loc_records_raw.json"
df.to_csv(csv_path, index=False, encoding="utf-8")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(kept, f, ensure_ascii=False, indent=2)

print(f"✓ Saved {len(df)} records to {csv_path}")
print(f"✓ Images:   {IMG_DIR}")
print(f"✓ Raw dump: {json_path}")

✓ Saved 265 records to /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/loc/loc_records.csv
✓ Images:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/loc/images
✓ Raw dump: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/loc/loc_records_raw.json


In [6]:
# ---- Stats ----
print(f"Total records kept: {len(df)}")
print(f"Records with non-empty date:        {(df['date'] != '').sum()}")
print(f"Records with non-empty description: {(df['description'] != '').sum()}")
print()
print("Title length summary (chars):")
print(df["title"].str.len().describe().round(1).to_string())
print()
print("Sample titles:")
for t in df["title"].head(5):
    print(f"  - {t[:100]}")

Total records kept: 265
Records with non-empty date:        251
Records with non-empty description: 221

Title length summary (chars):
count    265.0
mean      23.9
std       14.0
min       14.0
25%       18.0
50%       20.0
75%       20.0
max      105.0

Sample titles:
  - [Unidentified Production: Unidentified Portrait]
  - [Unidentified Production A: Unidentified Character]
  - [Unidentified Production: Unidentified "Men + Women"]
  - [Unidentified Production A: Unidentified Characters 2]
  - Unidentified Portraits


## Fallback — Wikimedia Commons (only run if LoC pulled too few)

If LoC returned fewer than ~200 usable records, run the cell below to top up from Wikimedia Commons' `Category:Unidentified_people`. This is a smaller but still thematic source. The output is appended to the same CSV.

**Skip this cell** if the LoC pull already gave you ≥ 250 records.

In [7]:
# ---- Wikimedia Commons fallback (optional) ----
if len(df) >= 250:
    print(f"LoC already has {len(df)} records (≥250) — skipping Wikimedia fallback.")
else:
    print(f"LoC only got {len(df)} records — topping up from Wikimedia Commons...")
    WMC_API = "https://commons.wikimedia.org/w/api.php"
    need = 260 - len(df)
    cm_params = {
        "action": "query", "format": "json", "list": "categorymembers",
        "cmtitle": "Category:Unidentified_people", "cmlimit": min(500, need * 2), "cmtype": "file",
    }
    r = session.get(WMC_API, params=cm_params, timeout=30)
    members = r.json().get("query", {}).get("categorymembers", [])
    print(f"  found {len(members)} files; fetching image info for first {need}...")
    extra = []
    for m in members[:need]:
        ii = session.get(WMC_API, params={
            "action": "query", "format": "json", "titles": m["title"],
            "prop": "imageinfo", "iiprop": "url|extmetadata", "iiurlwidth": 400,
        }, timeout=20).json()
        pages = ii.get("query", {}).get("pages", {})
        for pid, page in pages.items():
            info = (page.get("imageinfo") or [{}])[0]
            thumb = info.get("thumburl") or info.get("url")
            if not thumb: continue
            meta = info.get("extmetadata", {})
            extra.append({
                "loc_id":      m["title"].replace("File:", "")[:60],
                "title":       meta.get("ObjectName", {}).get("value", m["title"]),
                "date":        meta.get("DateTimeOriginal", {}).get("value", ""),
                "description": meta.get("ImageDescription", {}).get("value", ""),
                "subject":     "Wikimedia: Unidentified people",
                "location":    "",
                "permalink":   f"https://commons.wikimedia.org/wiki/{m['title']}",
                "image_url":   thumb,
            })
        polite_sleep(0.5)
    print(f"  pulled {len(extra)} Wikimedia records (download + merge step omitted — re-run the download cell on these if you want them appended).")

LoC already has 265 records (≥250) — skipping Wikimedia fallback.


## Notes for the report

1. **Why 'unidentified subjects' specifically.** This is a metadata field where the catalogue *itself acknowledges its own failure to know who or what the image depicts*. The same epistemic stance the thesis takes about AI — *the system trying to remember things it does not understand* — is already inscribed in the LoC catalogue as an administrative category. The project does not impose this framing; it inherits it.
2. **Why the smallest image, not the largest.** The thesis aesthetic specifies low-resolution, compressed, slightly-wrong imagery. Downloading the 400px thumbnail rather than the 2000px master scan is therefore a *design decision*, not a download-time optimisation.
3. **CC / Public domain.** Most LoC photographs are public domain or unrestricted — confirm per-record at the permalink before publishing. Wikimedia fallback files are CC-BY-SA where applicable; credit in the bibliography.